# MNIST — Dense Only (Baseline)

**Propósito**: Demostrar que la infraestructura FFI/TUI funciona correctamente usando una arquitectura puramente densa (sin convoluciones). Si este notebook anda, cualquier error en los notebooks `conv_*` es un problema de la **implementación ML de Conv2d**, no del stack FFI/TUI.

**Arquitectura**: Dense(784→256) → Dense(256→10), sin activaciones, MSE loss.

## 1. Setup & Descarga MNIST

In [ ]:
import os, struct, gzip, shutil, urllib.request, subprocess, time, datetime
import numpy as np

NWORKERS = 3
NSERVERS = 2
BASE_WORKER_PORT = 50000
BASE_SERVER_PORT = 40000
WORKER_ADDRS = [f'worker-{i}:{BASE_WORKER_PORT + i}' for i in range(NWORKERS)]
SERVER_ADDRS = [f'server-{i}:{BASE_SERVER_PORT + i}' for i in range(NSERVERS)]

DOCKER_DIR   = os.path.abspath('../../docker')
COMPOSE_FILE = os.path.abspath('../../compose.yaml')

URLS = {
    'train_images': 'https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz',
    'train_labels': 'https://storage.googleapis.com/cvdf-datasets/mnist/train-labels-idx1-ubyte.gz',
    'test_images':  'https://storage.googleapis.com/cvdf-datasets/mnist/t10k-images-idx3-ubyte.gz',
    'test_labels':  'https://storage.googleapis.com/cvdf-datasets/mnist/t10k-labels-idx1-ubyte.gz',
}

RAW_DIR          = 'data/mnist_raw'
TRAIN_SAMPLES_BIN = 'data/mnist_train_samples.bin'
TRAIN_LABELS_BIN  = 'data/mnist_train_labels.bin'
TEST_SAMPLES_BIN  = 'data/mnist_test_samples.bin'
TEST_LABELS_BIN   = 'data/mnist_test_labels.bin'
SAFETENSORS_PATH  = 'data/mnist_dense_only_model.safetensors'

X_SIZE = 784
Y_SIZE = 10

def download(name, url):
    os.makedirs(RAW_DIR, exist_ok=True)
    gz  = os.path.join(RAW_DIR, name + '.gz')
    raw = os.path.join(RAW_DIR, name)
    if os.path.exists(raw):
        print(f'  already exists: {raw}')
        return raw
    print(f'  downloading {name}...')
    urllib.request.urlretrieve(url, gz)
    with gzip.open(gz, 'rb') as fi, open(raw, 'wb') as fo:
        shutil.copyfileobj(fi, fo)
    os.remove(gz)
    return raw

def read_images(path):
    with open(path, 'rb') as f:
        _, n, rows, cols = struct.unpack('>IIII', f.read(16))
        raw = f.read(n * rows * cols)
    ppi = rows * cols
    return [[b / 255.0 for b in raw[i * ppi:(i + 1) * ppi]] for i in range(n)]

def read_labels(path):
    with open(path, 'rb') as f:
        _, n = struct.unpack('>II', f.read(8))
        return list(f.read(n))

def write_split_bins(images, labels, samples_path, labels_path):
    os.makedirs(os.path.dirname(samples_path), exist_ok=True)
    os.makedirs(os.path.dirname(labels_path),  exist_ok=True)
    with open(samples_path, 'wb') as fs, open(labels_path, 'wb') as fl:
        for img, lbl in zip(images, labels):
            one_hot = [0.0] * Y_SIZE
            one_hot[lbl] = 1.0
            fs.write(struct.pack(f'{X_SIZE}f', *img))
            fl.write(struct.pack(f'{Y_SIZE}f', *one_hot))
    print(f'  wrote {len(images)} samples -> {samples_path}')

for name, url in URLS.items():
    download(name, url)

if not (os.path.exists(TRAIN_SAMPLES_BIN) and os.path.exists(TRAIN_LABELS_BIN)):
    write_split_bins(
        read_images(os.path.join(RAW_DIR, 'train_images')),
        read_labels(os.path.join(RAW_DIR, 'train_labels')),
        TRAIN_SAMPLES_BIN, TRAIN_LABELS_BIN,
    )
else:
    print('training set already exists')

if not (os.path.exists(TEST_SAMPLES_BIN) and os.path.exists(TEST_LABELS_BIN)):
    write_split_bins(
        read_images(os.path.join(RAW_DIR, 'test_images')),
        read_labels(os.path.join(RAW_DIR, 'test_labels')),
        TEST_SAMPLES_BIN, TEST_LABELS_BIN,
    )
else:
    print('test set already exists')

## 2. Arquitectura — Dense Only

| Layer | Config | Params |
|---|---|---|
| Dense | 784 → 256, sin activación | 200 960 |
| Dense | 256 → 10, sin activación | 2 570 |

Total: **203 530 parámetros**

In [ ]:
DENSE_LAYERS = [
    {'size': 256, 'act': None},
    {'size': 10,  'act': None},
]

## 3. Start Workers & Servers

In [ ]:
import getpass

sudo_password = getpass.getpass('sudo password: ')

subprocess.run(
    ['python3', f'{DOCKER_DIR}/gen_compose.py'],
    env={**os.environ, 'WORKERS': str(NWORKERS), 'SERVERS': str(NSERVERS), 'RELEASE': 'true'},
    check=True,
)

subprocess.run(
    ['sudo', '-S', '-E', 'python3', f'{DOCKER_DIR}/fill_hosts.py'],
    input=sudo_password + '\n',
    text=True,
    env={**os.environ, 'WORKERS': str(NWORKERS), 'SERVERS': str(NSERVERS)},
    check=True,
)

subprocess.run(
    ['docker', 'compose', '-f', COMPOSE_FILE, 'up', '--build', '-d', '--remove-orphans'],
    check=True,
)

time.sleep(3)
print('All entities running.')

## 4. Entrenamiento Distribuido

Config: MSE, lr=0.01, SparseSerializer(r=0.95), BarrierSync + BlockingStore, 50 epochs, offline_epochs=2.

In [ ]:
import orchestra
from orchestra import Sequential, orchestrate
from orchestra.arch import Dense
from orchestra.initialization import Kaiming
from orchestra.datasets import LocalDataset
from orchestra.optimizers import GradientDescent
from orchestra.sync import BarrierSync
from orchestra.store import BlockingStore
from orchestra.loss_fns import Mse
from orchestra.serializer import SparseSerializer

model = Sequential([
    Dense(l['size'], Kaiming(), None)
    for l in DENSE_LAYERS
])

training = orchestra.parameter_server(
    worker_addrs=WORKER_ADDRS,
    server_addrs=SERVER_ADDRS,
    dataset=LocalDataset(TRAIN_SAMPLES_BIN, TRAIN_LABELS_BIN, x_size=784, y_size=10),
    optimizer=GradientDescent(lr=0.01),
    loss_fn=Mse(),
    sync=BarrierSync(),
    store=BlockingStore(),
    max_epochs=50,
    batch_size=128,
    offline_epochs=2,
    seed=42,
    early_stopping_tolerance=1e-6,
)

training_success = False
trained = None

try:
    print('Starting training session...')
    session = orchestrate(model, training)
    trained = session.wait()
    training_success = True
    print(f'Training complete — {len(trained.weights())} parameters')
    trained.save_safetensors(SAFETENSORS_PATH)
    print(f'Model saved to {SAFETENSORS_PATH}')
except Exception as e:
    print(f'[ERROR] Training failed: {e}')

## 5. Logs de Docker → /tmp/

In [ ]:
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
LOG_PATH = f'/tmp/mnist_dense_only_{timestamp}.log'

with open(LOG_PATH, 'w') as f:
    f.write(f'=== MNIST Dense Only — Docker Logs ===\n')
    f.write(f'=== Collected at {datetime.datetime.now().isoformat()} ===\n\n')
    f.write(f'=== training_success={training_success} ===\n\n')
    for name in [f'server-{i}' for i in range(NSERVERS)] + [f'worker-{i}' for i in range(NWORKERS)]:
        result = subprocess.run(
            ['docker', 'logs', '--timestamps', name],
            capture_output=True, text=True,
        )
        f.write(f'\n{"="*60}\n=== {name} ===\n{"="*60}\n')
        f.write(result.stdout)
        if result.stderr:
            f.write(result.stderr)

print(f'Docker logs saved to: {LOG_PATH}')

## 6. Evaluación con PyTorch

Carga los pesos del `.safetensors` en un modelo PyTorch equivalente y evalúa en el test set de MNIST.

Convención de pesos ONO: Dense almacena `[in, out]` — PyTorch `Linear` espera `[out, in]`, se transpone al cargar.

In [ ]:
if not training_success:
    print('Training failed — skipping evaluation.')
else:
    import torch
    import torch.nn as nn
    from safetensors.torch import load_file

    class OnoNet(nn.Module):
        def __init__(self, sizes):
            super().__init__()
            self.linears = nn.ModuleList([
                nn.Linear(sizes[i], sizes[i + 1]) for i in range(len(sizes) - 1)
            ])
        def forward(self, x):
            for linear in self.linears:
                x = linear(x)
            return x

    sizes = [X_SIZE] + [l['size'] for l in DENSE_LAYERS]
    net = OnoNet(sizes)
    state_dict = load_file(SAFETENSORS_PATH)

    with torch.no_grad():
        for j, linear in enumerate(net.linears):
            linear.weight.copy_(state_dict[f'layer_{j}.weight'].T)
            linear.bias.copy_(state_dict[f'layer_{j}.bias'])

    net.eval()
    x_raw = np.fromfile(TEST_SAMPLES_BIN, dtype=np.float32).reshape(-1, X_SIZE)
    y_raw = np.fromfile(TEST_LABELS_BIN,  dtype=np.float32).reshape(-1, Y_SIZE)
    x_test = torch.tensor(x_raw)
    y_test = torch.tensor(y_raw.argmax(axis=1), dtype=torch.long)

    with torch.no_grad():
        preds = net(x_test).argmax(dim=1)
        acc   = (preds == y_test).float().mean().item() * 100

    print(f'Test accuracy: {acc:.2f}%')

## 7. Stop Workers & Servers

In [ ]:
print('Stopping all entities...')
subprocess.run(
    ['docker', 'compose', '-f', COMPOSE_FILE, 'down'],
    check=True,
)
print('Done.')